# 06 Agent 端到端体检

**测什么**: 按 `.env` 装配运行时(checkpointer + store), 跑一次完整 graph 调用,
看是否产出答复、用到哪些工具、是否停在 HITL 询问。

**判定**: 产出非空 `final_answer` 为 PASS; 停在 interrupt 或报错为 FAIL(附停在哪个节点)。
无 LLM 密钥时 SKIP——因为此时必然停在 degrade 询问, 跑一遍要花掉数分钟而无诊断增量。

**前置**: 02 册(LLM)通过。检索/记忆可缺, 缺失时答复质量下降但不阻塞。


In [1]:
import asyncio, os, sys
from pathlib import Path

for cand in (Path.cwd(), *Path.cwd().parents):
    if (cand / "nbkit.py").is_file():
        NB_DIR = cand
        break
    if (cand / "tests_ipynb" / "nbkit.py").is_file():
        NB_DIR = cand / "tests_ipynb"
        break
else:
    raise RuntimeError("未找到 nbkit.py")

sys.path.insert(0, str(NB_DIR))

from nbkit import Checks, bootstrap

ROOT = bootstrap()
checks = Checks("06 端到端体检")

print("解释器  :", sys.executable)
print("仓库根  :", ROOT)
print("HF_HOME :", os.getenv("HF_HOME", "(未设置)"))


解释器  : F:\Anaconda_env\lawApp_langGraph\python.exe
仓库根  : E:\LangChain_LawAgent-main
HF_HOME : E:\huggingface_cache


In [2]:
import json, time
from lawApp_LangGraph.config import settings as s

QUERY = "我打算离婚, 房子是婚后买的, 首付我出得多, 怎么分割?"
key_ok = bool((s.deepseek_api_key or "").strip())
RUN_E2E_WITHOUT_KEY = False
RUN = key_ok or RUN_E2E_WITHOUT_KEY

if not key_ok and not RUN_E2E_WITHOUT_KEY:
    checks.fail("LLM 凭据", "DEEPSEEK_API_KEY 为空 → 图必然停在 degrade 询问, 本册跳过")
print("query  :", QUERY)
print("实跑   :", RUN)

query  : 我打算离婚, 房子是婚后买的, 首付我出得多, 怎么分割?
实跑   : True


## 1. 装配运行时(checkpointer + store)

In [3]:
if RUN:
    from lawApp_LangGraph import runtime

    t0 = time.time()
    await asyncio.wait_for(runtime.setup_runtime(), 180)
    checks.ok(
        "运行时装配",
        f"{time.time() - t0:.1f}s | checkpoint_backend={runtime.checkpoint_backend}",
    )
    if runtime.checkpoint_backend != "postgres":
        checks.skip(
            "Postgres 持久化",
            f"当前降级为 {runtime.checkpoint_backend} → 进程重启会话即丢(见 03 册)",
        )
    else:
        checks.ok("Postgres 持久化", "checkpointer + store 均为 Postgres")
else:
    checks.skip("运行时装配", "无密钥")

F:\Anaconda_env\lawApp_langGraph\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


MCP server 不可用, 跳过挂载: unhandled errors in a TaskGroup (1 sub-exception)


[PASS] 运行时装配 | 25.0s | checkpoint_backend=inmemory
[SKIP] Postgres 持久化 | 当前降级为 inmemory → 进程重启会话即丢(见 03 册)


## 2. 跑一次完整调用

In [4]:
if RUN:
    from lawApp_LangGraph.FastAPI.utils import extract_interrupt, graph_config
    from lawApp_LangGraph.LangGraph_lawApp import get_graph

    graph = get_graph()
    config = graph_config("nb-e2e-check")
    t0 = time.time()
    try:
        state = await asyncio.wait_for(graph.ainvoke({"query": QUERY}, config=config), 600)
    except asyncio.TimeoutError:
        checks.fail("graph.ainvoke 完成", "超过 600s 未返回")
        state = None
    except Exception as e:
        checks.fail("graph.ainvoke 完成", f"{type(e).__name__}: {str(e)[:200]}")
        state = None
    else:
        dt = time.time() - t0
        plan = state.get("plan") or []
        calls = [getattr(t, "tool_name", "") for t in (state.get("tool_calls") or [])]
        print(f"耗时        : {dt:.1f}s")
        print(f"计划步骤    : {[(x.step_id, x.tool_name, x.status) for x in plan]}")
        print(f"工具调用    : {calls}")
        print(f"法条/案例   : {len(state.get('law_results') or [])} 条 / {len(state.get('rag_documents') or [])} 条")
        print(f"error       : {str(state.get('error'))[:120]}")
        print(f"error_streak: {state.get('error_streak')}")

        snapshot = await graph.aget_state(config)
        itr = extract_interrupt(snapshot)
        answer = (state.get("final_answer") or "").strip()

        if answer:
            checks.ok("产出答复", f"{len(answer)} 字 | 工具={calls or '无'}")
        elif itr:
            checks.fail(
                "产出答复",
                f"停在 interrupt: type={itr.get('type')} → "
                f"{(itr.get('message') or itr.get('question') or '')[:120]}",
            )
        else:
            checks.fail("产出答复", "final_answer 为空且无 interrupt")

        degraded = [r for r in (state.get("reasoning") or []) if "失败" in str(r)]
        checks.expect(
            not degraded,
            "计划环节未走兜底",
            ok_detail="LLM 生成的计划与重规划正常",
            fail_detail="走了硬编码兜底: " + " | ".join(str(x) for x in degraded)[:160]
            + " → 根因见 02 册(structured output 不可用)",
        )
        n_law = len(state.get("law_results") or [])
        n_rag = len(state.get("rag_documents") or [])
        checks.expect(
            n_law + n_rag > 0,
            "答复有检索依据",
            ok_detail=f"法条 {n_law} 条 / 案例 {n_rag} 条",
            fail_detail=f"法条 {n_law} 条 / 案例 {n_rag} 条 → 答复为纯 LLM 产出, 无引用来源(根因见 03 册)",
        )
else:
    checks.skip("graph.ainvoke 完成", "无密钥")

耗时        : 15.2s
计划步骤    : []
工具调用    : []
法条/案例   : 0 条 / 0 条
error       : None
error_streak: 0
[FAIL] 产出答复 | 停在 interrupt: type=clarify → 这套房子是婚后哪一年买的？首付具体是谁出的、出了多少，有没有转账记录或凭证？ 对方现在对这套房子是什么态度——是想分一半、要房子，还是愿意谈补偿？ 你们有没有孩子？如果有，孩子现在跟谁生活、抚养问题谈得怎么样？
[PASS] 计划环节未走兜底 | LLM 生成的计划与重规划正常
[FAIL] 答复有检索依据 | 法条 0 条 / 案例 0 条 → 答复为纯 LLM 产出, 无引用来源(根因见 03 册)


## 3. 答复内容抽样

In [5]:
if RUN and state and (state.get("final_answer") or "").strip():
    print(state["final_answer"][:1200])
else:
    print("(无答复可展示)")

(无答复可展示)


## 4. 清理

In [6]:
if RUN:
    from lawApp_LangGraph import runtime
    from lawApp_LangGraph.db import close_pool

    await runtime.teardown_runtime()
    await close_pool()
    print("运行时已清理")

MCP 会话关闭异常: As of langchain-mcp-adapters 0.1.0, MultiServerMCPClient cannot be used as a context manager (e.g., async with MultiServerMCPClient(...)). Instead, you can do one of the following:
1. client = MultiServerMCPClient(...)
   tools = await client.get_tools()
2. client = MultiServerMCPClient(...)
   async with client.session(server_name) as session:
       tools = await load_mcp_tools(session)


运行时已清理


## 汇总

In [7]:
print(checks.report())


06 端到端体检 — 汇总
✓ 运行时装配       PASS  25.0s | checkpoint_backend=inmemory
- Postgres 持久化  SKIP  当前降级为 inmemory → 进程重启会话即丢(见 03 册)
✗ 产出答复         FAIL  停在 interrupt: type=clarify → 这套房子是婚后哪一年买的？首付具体是谁出的、出了多少，有没有转账记录或凭证？ 对方现在对这套房子是什么态度——是想分一半、要房子，还是愿意谈补偿？ 你们有没有孩子？如果有，孩子现在跟谁生活、抚养问题谈得怎么样？
✓ 计划环节未走兜底 PASS  LLM 生成的计划与重规划正常
✗ 答复有检索依据   FAIL  法条 0 条 / 案例 0 条 → 答复为纯 LLM 产出, 无引用来源(根因见 03 册)
合计: PASS 2 / FAIL 2 / SKIP 1

HAS FAILURES:
  ✗ 产出答复 | 停在 interrupt: type=clarify → 这套房子是婚后哪一年买的？首付具体是谁出的、出了多少，有没有转账记录或凭证？ 对方现在对这套房子是什么态度——是想分一半、要房子，还是愿意谈补偿？ 你们有没有孩子？如果有，孩子现在跟谁生活、抚养问题谈得怎么样？
  ✗ 答复有检索依据 | 法条 0 条 / 案例 0 条 → 答复为纯 LLM 产出, 无引用来源(根因见 03 册)
HAS FAILURES
